# Volume 5, Chapter 6: The zeros of zeta (Python companion)

Exercises first, solutions at the end. We list the zeros up to height $200$ and compare their number with the Riemann–von Mangoldt formula (Theorem 6.9), check by sign changes that they all lie on the critical line, test Hadamard's constant $B$ (Exercises 6.1 and 6.2), Stirling's formula in the complex plane (Theorem 6.1) and Jensen's formula (Theorem 6.3), and end with root-finding, the computer-science side of the search for zeros. Requirements: numpy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P6.1 (★) Counting.** List the ordinates $0<\gamma\le200$ with `mpmath.zetazero`, and compare $N(T)$ with $\frac{\vartheta(T)}\pi+1$ and with the main term $\frac T{2\pi}\log\frac T{2\pi e}+\frac78$. How large does $S(T)$ get, and what is its average?

**P6.2 (★★) All on the line.** Hardy's $Z(t)$ is real (Volume 4, Chapter 10). Count its sign changes on $]0,200]$ and compare with Theorem 6.9. What exactly does the comparison prove, and what is missing?

**P6.3 (★) Hadamard's constant.** Compute $B=\frac12\log4\pi-1-\frac\gamma2$ and the partial sums of $\sum_\rho\operatorname{Re}\frac1\rho$ over $|\gamma|\le T$, assuming the zeros found are on the line. Estimate the tail with Theorem 6.9.

**P6.4 (★) Stirling off the axis.** Compare $\log\Gamma(s)$ with $(s-\frac12)\log s-s+\frac12\log2\pi$ for a few complex $s$, and the error with $\frac1{12|s|}$. Check $\psi_\Gamma(\frac12)=-\gamma-2\log2$.

**P6.5 (★) Jensen at work.** Check Jensen's formula for $f(z)=\xi(\frac12+z)$ on the circle $|z|=30$.

**P6.6 (★★) CS application: finding zeros.** Locate the first $20$ zeros to $10^{-10}$ from the sign changes of P6.2, by bisection and by the Illinois variant of regula falsi. Count the evaluations of $Z$.

## Solutions

### P6.1

In [ ]:
import math
import numpy as np
import mpmath as mp
mp.mp.dps = 20

zeros = []
n = 1
while True:
    g = float(mp.zetazero(n).imag)
    if g > 200: break
    zeros.append(g); n += 1
zeros = np.array(zeros)
def N(T): return int(np.searchsorted(zeros, T, side='right'))
def vartheta(T): return float(mp.siegeltheta(T))
def main_term(T): return T / (2 * math.pi) * math.log(T / (2 * math.pi * math.e)) + 7 / 8
print(f"{len(zeros)} zeros with 0 < gamma <= 200; the first five: {np.round(zeros[:5], 4)}")
for T in (20, 50, 100, 150, 200):
    S = N(T) - vartheta(T) / math.pi - 1
    print(f"T = {T:3d}: N(T) = {N(T):2d}, main term = {main_term(T):7.3f}, S(T) = {S:+.3f}")
Ts = np.linspace(10, 200, 20000)
S_vals = np.array([N(T) - vartheta(T) / math.pi - 1 for T in Ts])
print(f"on [10, 200]: min S = {S_vals.min():+.3f}, max S = {S_vals.max():+.3f}, mean S = {S_vals.mean():+.4f}")

$S(T)$ oscillates around $0$, with $|S(T)|<1$ on this range and a mean very close to $0$ (Littlewood proved that $\int_0^TS(t)\,dt=O(\log T)$). The bound $O(\log T)$ of Theorem 6.9 is far from what one sees: the largest values known, near $10^{28}$, are only about $3$. Selberg showed that $S(T)$ is typically of size $\frac1\pi\sqrt{\frac12\log\log T}$, Gaussian in distribution.

### P6.2

In [ ]:
def Z(t): return float(mp.siegelz(t))
grid = np.arange(0.5, 200.0, 0.05)
vals = np.array([Z(t) for t in grid])
changes = grid[:-1][np.sign(vals[:-1]) != np.sign(vals[1:])]
print(f"sign changes of Z on ]0, 200]: {len(changes)}")
print(f"vartheta(200)/pi + 1 = {vartheta(200) / math.pi + 1:.3f}")
print("so all", len(changes), "zeros up to height 200 lie on the critical line, provided |S(200)| < 1 (Turing's method makes this rigorous)")

$79$ sign changes means at least $79$ zeros on the line with $0<\gamma\le200$, each of odd multiplicity. Theorem 6.9 gives $N(200)=79.193+S(200)$, and $N(200)$ is an integer; if $|S(200)|<1$ then $N(200)\le80$, and since the count of zeros off the line comes in pairs ($\rho$ and $1-\overline\rho$), $N(200)=79$ and every zero up to $200$ is on the line and simple. The gap is the bound $|S(200)|<1$, which needs an explicit version of the theorem: Turing's method supplies it from the average of $S$ near $T$, using only computed values of $Z$. (The grid of step $0.05$ could also miss two close sign changes; the count matching the formula shows it did not.)

### P6.3

In [ ]:
gamma_E = float(mp.euler)
B = 0.5 * math.log(4 * math.pi) - 1 - gamma_E / 2
print(f"B = {B:.7f}")
def partial(T):
    g = zeros[zeros <= T]
    return float(np.sum(1.0 / (0.25 + g**2)))          # each pair rho, conj(rho) gives 2 * (1/2)/|rho|^2
for T in (50, 100, 200):
    tail = mp.quad(lambda t: math.log(t / (2 * math.pi)) / (2 * math.pi) / (0.25 + t**2), [T, mp.inf])
    print(f"sum over gamma <= {T}: {partial(T):.6f};  with the tail estimated from N(T): {partial(T) + float(tail):.6f};  -B = {-B:.6f}")

Each pair $\rho,\overline\rho$ on the line contributes $2\cdot\frac{1/2}{1/4+\gamma^2}$. The partial sums converge slowly (the tail beyond $T$ is about $\frac{\log(T/2\pi)}{2\pi T}$), but with the tail estimated from the density of zeros the agreement with $-B=0.0230957$ is excellent. This is a numerical test of Hadamard's product and of the Riemann hypothesis at once: a zero off the line would change $\operatorname{Re}\frac1\rho$.

### P6.4

In [ ]:
for s in (mp.mpc(0.25, 10), mp.mpc(0.25, 50), mp.mpc(-3, 20), mp.mpc(5, 1)):
    exact = mp.loggamma(s)
    approx = (s - 0.5) * mp.log(s) - s + 0.5 * mp.log(2 * mp.pi)
    print(f"s = {complex(s)}: |log Gamma - Stirling| = {float(abs(exact - approx)):.2e}, 1/(12|s|) = {1 / (12 * float(abs(s))):.2e}")
print("psi(1/2) =", float(mp.digamma(0.5)), " -gamma - 2 log 2 =", -gamma_E - 2 * math.log(2))

The error is $\frac1{12s}$ to first order, the next term of Stirling's series (Volume 4, Chapter 11), and it stays small even at $s=-3+20i$, close to the negative axis in angle but far from it in distance: Theorem 6.1 needs only a sector. Legendre's duplication gives $\psi_\Gamma(\frac12)$ exactly.

### P6.5

In [ ]:
def xi(s): return 0.5 * s * (s - 1) * mp.pi**(-s / 2) * mp.gamma(s / 2) * mp.zeta(s)
R = 30
c = mp.mpf(0.5)
mean_log = mp.quad(lambda th: mp.log(abs(xi(c + R * mp.expj(th)))), [0, mp.pi / 2, mp.pi, 3 * mp.pi / 2, 2 * mp.pi]) / (2 * mp.pi)
inside = [g for g in zeros if g < R]
lhs = mp.log(abs(xi(c))) + 2 * sum(mp.log(R / g) for g in inside)
print(f"zeros of xi(1/2 + z) in |z| < {R}: {2 * len(inside)} (at +-i*gamma, gamma = {np.round(inside, 3)})")
print(f"Jensen: log|xi(1/2)| + sum log(R/|z_k|) = {float(lhs):.8f};  mean of log|xi| on the circle = {float(mean_log):.8f}")

The six zeros $\frac12\pm i\gamma$ with $\gamma<30$ are inside, and the two sides agree to the eight digits shown. The average of $\log|\xi|$ on the circle is pulled up by the right and left extremities, where $\xi$ is huge (it grows like $e^{\frac12|s|\log|s|}$ on the real axis); the zeros pay for this growth.

### P6.6

In [ ]:
calls = {'n': 0}
def Zc(t):
    calls['n'] += 1
    return float(mp.siegelz(t))
def bisect(a, b, tol=1e-10):
    fa = Zc(a)
    while b - a > tol:
        m = (a + b) / 2; fm = Zc(m)
        if np.sign(fm) == np.sign(fa): a, fa = m, fm
        else: b = m
    return (a + b) / 2
def secant_illinois(a, b, tol=1e-10):
    fa, fb = Zc(a), Zc(b); side = 0
    while abs(b - a) > tol:
        c = (a * fb - b * fa) / (fb - fa); fc = Zc(c)
        if fc * fb > 0:
            b, fb = c, fc
            if side == -1: fa /= 2
            side = -1
        elif fa * fc > 0:
            a, fa = c, fc
            if side == 1: fb /= 2
            side = 1
        else:
            return c
    return (a + b) / 2
for name, method in (("bisection", bisect), ("Illinois (regula falsi)", secant_illinois)):
    calls['n'] = 0
    roots = [method(a, a + 0.05) for a in changes[:20]]
    err = max(abs(r - g) for r, g in zip(roots, zeros[:20]))
    print(f"{name:24s}: {calls['n'] / 20:5.1f} evaluations of Z per zero, max error {err:.1e}")

Bisection halves the interval at each step, so $0.05\to10^{-10}$ costs $\log_2(5\cdot10^8)\approx29$ evaluations. Regula falsi uses the values, not only the signs; the Illinois modification (halve the value kept at a stale endpoint) restores superlinear convergence, and $7$ or $8$ evaluations suffice. At large heights each evaluation of $Z$ by the Riemann–Siegel formula costs about $\sqrt{t/2\pi}$ terms, so economy matters: the Odlyzko–Schönhage algorithm evaluates $Z$ at many points at once with fast Fourier transforms, which is how billions of zeros near $10^{22}$ were computed.